# SA-DMAE 전처리 노트북
BraTS 2021 + UCSF-PDGM 3D NIfTI → 2.5D 슬라이스 `.pt` 파일 변환

**출력 형식:** `(3, 3, 224, 224)` float32 tensor  
**채널 순서:** T1ce / T2 / FLAIR  
**슬라이스:** (center_z-1, center_z, center_z+1) — 종양 ROI 기준

| 데이터셋 | 케이스 수 | 비고 |
|----------|-----------|------|
| BraTS 2021 | 1,251 | seg 파일로 tumor center 추출 |
| UCSF-PDGM | 323 | T2 없는 178개 스킵 |

In [ ]:
# ── Cell 1: Google Drive 마운트 ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: 패키지 설치 ───────────────────────────────────────────────────
!pip install nibabel tqdm -q

In [ ]:
# ── Cell 3: 경로 설정 (여기만 수정) ──────────────────────────────────────
import os

# ── 입력 경로 ──
BRATS_ROOT = '/content/drive/MyDrive/BraTS2021'       # ← 실제 경로로 수정
UCSF_ROOT  = '/content/drive/MyDrive/UCSF-PDGM'      # ← 실제 경로로 수정

# ── 출력 경로 ──
BRATS_OUT  = '/content/drive/MyDrive/SA_DMAE_slices/brats'  # ← 원하는 경로로 수정
UCSF_OUT   = '/content/drive/MyDrive/SA_DMAE_slices/ucsf'   # ← 원하는 경로로 수정

# ── 슬라이스 설정 ──
N_SLICES    = 3
TARGET_SIZE = (224, 224)

os.makedirs(BRATS_OUT, exist_ok=True)
os.makedirs(UCSF_OUT,  exist_ok=True)
print(f'BraTS output : {BRATS_OUT}')
print(f'UCSF  output : {UCSF_OUT}')

In [ ]:
# ── Cell 4: 공통 Slicer 함수 ─────────────────────────────────────────────
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from pathlib import Path


def load_volume(path):
    """NIfTI 파일 로드 → (H, W, D) float32 numpy array."""
    return nib.load(path).get_fdata(dtype=np.float32)


def normalize_volume(vol, low_pct=1.0, high_pct=99.0):
    """비영 voxel 기준 percentile 정규화 → [0, 1]."""
    nonzero = vol[vol > 0]
    if nonzero.size == 0:
        return vol
    lo = np.percentile(nonzero, low_pct)
    hi = np.percentile(nonzero, high_pct)
    if hi == lo:
        return np.zeros_like(vol)
    return (np.clip(vol, lo, hi) - lo) / (hi - lo)


def find_tumor_center_z(seg):
    """seg 마스크 기준 종양 Z-range 중앙 인덱스 반환."""
    tumor_mask   = seg > 0
    z_with_tumor = np.where(tumor_mask.any(axis=(0, 1)))[0]
    if len(z_with_tumor) == 0:
        return seg.shape[2] // 2   # fallback: 볼륨 중앙
    return int(z_with_tumor[len(z_with_tumor) // 2])


def extract_slice_stack(volumes, center_z, n_slices=3, target_size=(224, 224)):
    """연속 n_slices 장 추출 → (n_slices, C, H, W) tensor."""
    half     = n_slices // 2
    D        = volumes[0].shape[2]
    z_indices = [min(max(center_z + o, 0), D - 1) for o in range(-half, half + 1)]

    slices = []
    for z in z_indices:
        channels = [vol[:, :, z] for vol in volumes]
        s = torch.from_numpy(np.stack(channels, axis=0)).unsqueeze(0)  # (1, C, H, W)
        s = F.interpolate(s, size=target_size, mode='bilinear', align_corners=False)
        slices.append(s.squeeze(0))                                     # (C, H, W)
    return torch.stack(slices, dim=0)                                   # (n_slices, C, H, W)


def find_file(case_dir, pattern):
    """중첩 폴더에서 패턴에 맞는 첫 번째 파일 반환 (폴더 제외)."""
    matches = [p for p in Path(case_dir).rglob(pattern) if p.is_file()]
    return str(matches[0]) if matches else None


print('Slicer functions ready.')

In [ ]:
# ── Cell 5: BraTS 2021 전처리 ────────────────────────────────────────────
from tqdm import tqdm

BRATS_MODALITIES = ['t1ce', 't2', 'flair']   # 채널 순서 고정

brats_root = Path(BRATS_ROOT)
brats_out  = Path(BRATS_OUT)
all_cases  = sorted([d for d in brats_root.iterdir() if d.is_dir()])
done       = {p.stem for p in brats_out.glob('*.pt')}
to_run     = [c for c in all_cases if c.name not in done]

print(f'[BraTS] Total: {len(all_cases)}  |  Done: {len(done)}  |  Remaining: {len(to_run)}')

errors   = []
log_path = brats_out / 'preprocess_log.txt'

for case_dir in tqdm(to_run, desc='BraTS'):
    case_id = case_dir.name
    try:
        # 필요 파일 확인
        required = [f'{case_id}_{m}.nii.gz' for m in BRATS_MODALITIES] + [f'{case_id}_seg.nii.gz']
        missing  = [f for f in required if not (case_dir / f).exists()]
        if missing:
            raise FileNotFoundError(f'Missing: {missing}')

        # 로드 & 정규화 (T1ce / T2 / FLAIR)
        volumes = [
            normalize_volume(load_volume(str(case_dir / f'{case_id}_{m}.nii.gz')))
            for m in BRATS_MODALITIES
        ]
        seg      = load_volume(str(case_dir / f'{case_id}_seg.nii.gz'))
        center_z = find_tumor_center_z(seg)
        stack    = extract_slice_stack(volumes, center_z, N_SLICES, TARGET_SIZE)

        torch.save(stack.float(), brats_out / f'{case_id}.pt')
        with open(log_path, 'a') as f:
            f.write(f'OK  {case_id}  center_z={center_z}  shape={list(stack.shape)}\n')

    except Exception as e:
        errors.append((case_id, str(e)))
        with open(log_path, 'a') as f:
            f.write(f'ERR {case_id}  {e}\n')

print(f'\n[BraTS] Done. Success: {len(to_run) - len(errors)}  |  Errors: {len(errors)}')
if errors:
    for cid, err in errors:
        print(f'  {cid}: {err}')

In [ ]:
# ── Cell 6: UCSF-PDGM 전처리 ─────────────────────────────────────────────
# 파일 구조: {case_id}_nifti/{MOD_folder}/NIFTI/{file}.nii
# T2 없는 케이스는 스킵 (501개 중 178개)

ucsf_root = Path(UCSF_ROOT)
ucsf_out  = Path(UCSF_OUT)
all_cases = sorted([d for d in ucsf_root.iterdir() if d.is_dir()])
done      = {p.stem for p in ucsf_out.glob('*.pt')}
to_run    = [c for c in all_cases if c.name not in done]

print(f'[UCSF] Total: {len(all_cases)}  |  Done: {len(done)}  |  Remaining: {len(to_run)}')

errors    = []
skipped   = []
log_path  = ucsf_out / 'preprocess_log.txt'

for case_dir in tqdm(to_run, desc='UCSF-PDGM'):
    case_id = case_dir.name
    try:
        # 파일 탐색 (중첩 폴더 대응)
        t1ce  = find_file(case_dir, '*T1gad_bias.nii')
        t2    = find_file(case_dir, '*_T2.nii.gz')
        flair = find_file(case_dir, '*FLAIR_bias.nii')
        seg   = find_file(case_dir, '*tumor_segmentation.nii')

        # T2 없으면 스킵
        if t2 is None:
            skipped.append(case_id)
            with open(log_path, 'a') as f:
                f.write(f'SKIP {case_id}  reason=no_T2\n')
            continue

        # 필수 파일 확인
        missing = [name for name, path in [('T1ce', t1ce), ('FLAIR', flair), ('seg', seg)] if path is None]
        if missing:
            raise FileNotFoundError(f'Missing: {missing}')

        # 로드 & 정규화 (T1ce / T2 / FLAIR)
        volumes  = [normalize_volume(load_volume(p)) for p in [t1ce, t2, flair]]
        center_z = find_tumor_center_z(load_volume(seg))
        stack    = extract_slice_stack(volumes, center_z, N_SLICES, TARGET_SIZE)

        torch.save(stack.float(), ucsf_out / f'{case_id}.pt')
        with open(log_path, 'a') as f:
            f.write(f'OK  {case_id}  center_z={center_z}  shape={list(stack.shape)}\n')

    except Exception as e:
        errors.append((case_id, str(e)))
        with open(log_path, 'a') as f:
            f.write(f'ERR {case_id}  {e}\n')

print(f'\n[UCSF] Done. Success: {len(to_run) - len(errors) - len(skipped)}  |  Skipped(no T2): {len(skipped)}  |  Errors: {len(errors)}')
if errors:
    for cid, err in errors:
        print(f'  {cid}: {err}')

In [ ]:
# ── Cell 7: 전체 결과 요약 ────────────────────────────────────────────────
brats_pt = list(Path(BRATS_OUT).glob('*.pt'))
ucsf_pt  = list(Path(UCSF_OUT).glob('*.pt'))
total_pt = brats_pt + ucsf_pt

print('=== 전처리 결과 요약 ===')
print(f'BraTS .pt 파일 : {len(brats_pt)}')
print(f'UCSF  .pt 파일 : {len(ucsf_pt)}')
print(f'합계            : {len(total_pt)}')
print()

# 랜덤 샘플 shape/range 확인
import random, matplotlib.pyplot as plt

samples = random.sample(total_pt, min(4, len(total_pt)))
print('--- 샘플 검증 ---')
for p in samples:
    t = torch.load(p, map_location='cpu')
    src = 'BraTS' if 'BraTS' in p.stem else 'UCSF'
    print(f'[{src}] {p.stem}  shape={list(t.shape)}  min={t.min():.3f}  max={t.max():.3f}')

# 샘플 시각화
sample = torch.load(samples[0], map_location='cpu')   # (3, 3, 224, 224)
mod_names   = ['T1ce', 'T2', 'FLAIR']
slice_names = ['prev (Z-1)', 'center (Z)', 'next (Z+1)']

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
src = 'BraTS' if 'BraTS' in samples[0].stem else 'UCSF'
fig.suptitle(f'[{src}] {samples[0].stem}', fontsize=12)
for row, mod in enumerate(mod_names):
    for col, sname in enumerate(slice_names):
        ax = axes[row, col]
        ax.imshow(sample[col, row].numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'{mod} | {sname}', fontsize=8)
        ax.axis('off')
plt.tight_layout()
plt.savefig(str(Path(BRATS_OUT).parent / 'sample_verify.png'), dpi=100)
plt.show()
print('Verification complete.')